# Holiday calendars and business-day adjustment (reference)

**Deep dive:** listing registry calendars, constructing `HolidayCalendar`, and using `adjust` with every `BusinessDayConvention`. Includes cross-calendar reasoning (when two markets must both be open).


## Concept

- A **holiday calendar** defines which dates are **holidays** (non-business) for a market; weekends are typically non-business as well.
- **`adjust(date, convention, calendar)`** moves a **candidate** payment date onto a business day per **ISDA-style** rules: Following, Modified Following, Preceding, Modified Preceding, or Unadjusted.
- **Modified** variants avoid rolling into the **next month** (Following) or **previous month** (Preceding) when possible.
- **Multi-calendar** logic (e.g. cross-currency settlement) is “both markets open” in many desks: finstack’s Python `adjust` takes **one** calendar at a time, so you either pick the **stricter** calendar, call `is_business_day` on each, or rely on higher-level helpers elsewhere in the stack. Below we **compare** `usny` vs `target2` on the same dates.


## API walkthrough

### Listing calendars

`available_calendars()` returns every **string code** registered in the global calendar registry.


In [1]:
try:
    from finstack.core.dates import (
        available_calendars,
        HolidayCalendar,
        adjust,
        BusinessDayConvention,
    )
except ImportError as e:
    print("Import failed:", e)
else:
    cals = available_calendars()
    print(f"Available calendars: {cals}")
    print("count:", len(cals))


Available calendars: ['asx', 'auce', 'brbd', 'cato', 'chzh', 'cme', 'cnbe', 'defr', 'gblo', 'hkex', 'hkhk', 'jpto', 'jpx', 'nyse', 'nzau', 'sgsi', 'sifma', 'sse', 'target2', 'usny']
count: 20


### `HolidayCalendar` construction

Resolve by code (e.g. `usny`, `target2`, `gblo`). Use `is_holiday` / `is_business_day` for diagnostics.


In [2]:
from datetime import date

try:
    from finstack.core.dates import HolidayCalendar
except ImportError as e:
    print("Import failed:", e)
else:
    cal = HolidayCalendar("usny")
    print("HolidayCalendar:", cal)
    print("code:", cal.code)
    xmas = date(2024, 12, 25)
    print(f"is_holiday({xmas}) [USNY]:", cal.is_holiday(xmas))
    print(f"is_business_day({xmas}) [USNY]:", cal.is_business_day(xmas))


HolidayCalendar: usny
code: usny
is_holiday(2024-12-25) [USNY]: True
is_business_day(2024-12-25) [USNY]: False


### `adjust` with all `BusinessDayConvention` values

Same **Saturday** under each convention on **USNY**; then a **Modified Following** month-end example.


In [3]:
from datetime import date

try:
    from finstack.core.dates import adjust, BusinessDayConvention
except ImportError as e:
    print("Import failed:", e)
else:
    sat = date(2025, 1, 4)
    print("Input Saturday:", sat)
    for bdc in [
        BusinessDayConvention.UNADJUSTED,
        BusinessDayConvention.FOLLOWING,
        BusinessDayConvention.MODIFIED_FOLLOWING,
        BusinessDayConvention.PRECEDING,
        BusinessDayConvention.MODIFIED_PRECEDING,
    ]:
        adj = adjust(sat, bdc, "usny")
        print(f"  {bdc!s:22s} -> {adj}")
    may_end = date(2025, 5, 31)
    mf = adjust(may_end, BusinessDayConvention.MODIFIED_FOLLOWING, "usny")
    print("ModifiedFollowing on 2025-05-31 (USNY):", mf)


Input Saturday: 2025-01-04
  Unadjusted             -> 2025-01-04
  Following              -> 2025-01-06
  ModifiedFollowing      -> 2025-01-06
  Preceding              -> 2025-01-03
  ModifiedPreceding      -> 2025-01-03
ModifiedFollowing on 2025-05-31 (USNY): 2025-05-30


### Multi-calendar chain (logical joint business day)

For a **joint** check: a date is “good” for **both** USD and EUR if **both** calendars report a business day. Below we also show **different** `adjust` results when calendars disagree on holidays.


In [4]:
from datetime import date

try:
    from finstack.core.dates import HolidayCalendar, adjust, BusinessDayConvention
except ImportError as e:
    print("Import failed:", e)
else:
    us = HolidayCalendar("usny")
    eu = HolidayCalendar("target2")
    d = date(2025, 1, 2)
    joint = us.is_business_day(d) and eu.is_business_day(d)
    print(f"Date {d}: USNY business={us.is_business_day(d)}, T2 business={eu.is_business_day(d)}, joint={joint}")
    # Sequential adjustment illustration (not the same as composite union calendar — educational)
    step1 = adjust(d, BusinessDayConvention.FOLLOWING, us)
    step2 = adjust(step1, BusinessDayConvention.FOLLOWING, eu)
    print("Sequential FOLLOWING (USNY then T2):", d, "->", step1, "->", step2)


Date 2025-01-02: USNY business=True, T2 business=True, joint=True
Sequential FOLLOWING (USNY then T2): 2025-01-02 -> 2025-01-02 -> 2025-01-02


### Christmas and New Year adjustments

US and TARGET2 treat **Christmas** and **New Year** as holidays; `Following` rolls forward to the next business day in each market.


In [5]:
from datetime import date

try:
    from finstack.core.dates import adjust, BusinessDayConvention
except ImportError as e:
    print("Import failed:", e)
else:
    pairs = [
        ("Christmas 2024", date(2024, 12, 25)),
        ("New Year 2025", date(2025, 1, 1)),
    ]
    for label, raw in pairs:
        us = adjust(raw, BusinessDayConvention.FOLLOWING, "usny")
        t2 = adjust(raw, BusinessDayConvention.FOLLOWING, "target2")
        print(f"{label} raw={raw}  FOLLOWING usny={us}  target2={t2}")


Christmas 2024 raw=2024-12-25  FOLLOWING usny=2024-12-26  target2=2024-12-27
New Year 2025 raw=2025-01-01  FOLLOWING usny=2025-01-02  target2=2025-01-02


## Practical example

Pick a **roll date** that lands on a weekend; compare **Following** vs **Modified Following** for a **London** calendar (`gblo`).


In [6]:
from datetime import date

try:
    from finstack.core.dates import adjust, BusinessDayConvention
except ImportError as e:
    print("Import failed:", e)
else:
    roll = date(2025, 11, 1)  # Saturday
    fol = adjust(roll, BusinessDayConvention.FOLLOWING, "gblo")
    mf = adjust(roll, BusinessDayConvention.MODIFIED_FOLLOWING, "gblo")
    print("Candidate roll:", roll)
    print("Following (GBLO):", fol)
    print("ModifiedFollowing (GBLO):", mf)


Candidate roll: 2025-11-01
Following (GBLO): 2025-11-03
ModifiedFollowing (GBLO): 2025-11-03


## Takeaways

- **`available_calendars()`** is the registry index; **`HolidayCalendar(code)`** resolves a live calendar for queries.
- **`adjust`** implements standard **BDC** semantics; **Modified** rules prevent month jumps when a one-day roll would cross a month boundary.
- **Christmas / New Year** are ideal sanity checks when wiring a new calendar.
- For **two-currency** settlement, think explicitly about whether you need **union** (either holiday closes) or **intersection** semantics; Python `adjust` is **single-calendar** — combine **`is_business_day`** or use domain-specific helpers for joint logic.
